# English Speaking Practice Agent

OPIc / TOEIC Speaking 스타일 사진 묘사 연습을 위한 LangGraph 기반 교육 에이전트.

## 목표

1. AI가 사진을 생성하고, 사용자가 영어로 사진을 묘사하면
2. 음성을 전사(Whisper)하고, 웹 검색으로 참고자료를 수집한 뒤
3. 문법/어휘/문장구조를 교정하고, 모범답안을 제시한다
4. 사용자는 교정과 모범답안을 재생성할 수 있다

## Architecture

```
START
  │
  ▼
generate_image          ← GPT Image (gpt-image-1)
  │
  ▼
record_voice            ← interrupt() — 사용자 음성 입력 대기
  │
  ▼
transcribe              ← OpenAI Whisper (whisper-1)
  │
  ▼
search_references       ← Tavily Web Search (Tool)
  │
  ▼
correct_syntax          ← GPT-4o-mini + 검색 참고자료
  │
  ▼
recommend_ideal_answer  ← GPT-4o-mini (vision) + 이미지
  │
  ▼
ask_regenerate          ← interrupt() — 재생성 여부 확인
  │
  ├─ yes ──▶ correct_syntax (루프백)
  │
  └─ no ───▶ END
```

### 주요 설계 결정

| 결정 | 이유 |
|---|---|
| `interrupt()` for human-in-the-loop | Streamlit의 `st.audio_input`과 자연스럽게 연동. 그래프가 일시정지 → UI에서 입력 → `Command(resume=...)` 로 재개 |
| `Annotated[list, operator.add]` for corrections/recommendations | 재생성 시 덮어쓰기 대신 누적 → 사용자가 버전별 비교 가능 |
| `TavilySearch` 수동 호출 (ToolNode X) | 현재 State가 custom TypedDict이므로 `messages` 키 불필요. 검색 타이밍이 결정적(deterministic) |
| Conditional Edge (`ask_regenerate`) | `correct_syntax` → `recommend_ideal_answer` 루프만 재실행. 이미지 생성/전사/검색은 건너뜀 |

## Setup

In [ ]:
from openai import OpenAI
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langgraph.types import interrupt
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated, TypedDict
from dotenv import load_dotenv
import operator
import base64
import io
import os
from datetime import datetime

load_dotenv()

llm = init_chat_model("openai:gpt-4o-mini")
tavily = TavilySearch(max_results=3)

## State 정의

| 필드 | 타입 | 설명 |
|---|---|---|
| `image_dir` | `str` | 생성된 이미지 파일 경로 |
| `audio_bytes` | `bytes` | 사용자 음성 녹음 바이너리 (WAV) |
| `transcription` | `str` | Whisper 전사 결과 |
| `search_results` | `str` | Tavily 웹 검색 결과 (TOEIC/OPIc 참고자료) |
| `corrections` | `list[str]` | 문법/어휘 교정 이력 (`operator.add`로 누적) |
| `recommendations` | `list[str]` | 모범답안 이력 (`operator.add`로 누적) |
| `regenerate` | `bool` | 재생성 여부 (conditional edge 라우팅) |

In [ ]:
class State(TypedDict):
    image_dir: str
    audio_bytes: bytes
    transcription: str
    search_results: str
    corrections: Annotated[list[str], operator.add]
    recommendations: Annotated[list[str], operator.add]
    regenerate: bool

## Node 1: generate_image

OpenAI `gpt-image-1`으로 TOEIC Speaking/OPIc 스타일의 일상 장면 이미지를 생성한다.
3-6명의 인물, 명확한 행동, 구체적인 장소를 포함하도록 프롬프트를 구성.

In [ ]:
def generate_image(state: State):
    client = OpenAI()
    prompt = (
        "Create a realistic, everyday scene image for an English speaking test "
        "(TOEIC Speaking/OPIc-style picture description). The image should include "
        "3-6 people, clear actions, visible objects, and a specific location "
        "(e.g., cafe, office, park, airport, store). Add enough detail for a "
        "45-60 second spoken description: people's positions, clothing, expressions, "
        "interactions, background elements, and at least one notable event or contrast. "
        "Keep it natural, modern, and free of text/watermarks."
    )
    result = client.images.generate(
        model="gpt-image-1",
        prompt=prompt,
        quality="low",
        moderation="low",
        size="auto",
    )
    image_bytes = base64.b64decode(result.data[0].b64_json)
    current_timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    os.makedirs("images", exist_ok=True)
    image_dir = f"images/question_{current_timestamp}.jpg"

    with open(image_dir, "wb") as file:
        file.write(image_bytes)

    return {"image_dir": image_dir}

## Node 2: record_voice

LangGraph `interrupt()`를 사용해 그래프 실행을 일시정지한다.
Streamlit에서는 `st.audio_input`으로 녹음 후 `Command(resume=audio_bytes)`로 재개.

In [ ]:
def record_voice(state: State):
    audio_bytes = interrupt("Please record your voice describing the image.")
    return {"audio_bytes": audio_bytes}

## Node 3: transcribe

OpenAI Whisper (`whisper-1`)로 음성을 텍스트로 전사.
TOEIC/OPIc 문맥을 prompt에 포함해 전사 정확도를 높인다.

In [ ]:
def transcribe(state: State):
    client = OpenAI()
    audio_bio = io.BytesIO(state["audio_bytes"])
    audio_bio.name = "audio.wav"
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_bio,
        language="en",
        prompt=(
            "Transcribe the audio for an English speaking test "
            "(TOEIC Speaking/OPIc-style picture description). "
            "Capture all spoken words including hesitations and fillers."
        ),
    )
    return {"transcription": transcription.text}

## Node 4: search_references (Tool)

Tavily Web Search로 전사 내용 기반 TOEIC/OPIc 채점 기준, 어휘/표현 팁을 검색.
검색 결과는 `correct_syntax` 노드에서 교정의 근거자료로 활용된다.

In [ ]:
def search_references(state: State):
    query = (
        "TOEIC Speaking OPIc picture description "
        "scoring rubric vocabulary and expression tips "
        f"for: {state['transcription'][:200]}"
    )
    results = tavily.invoke({"query": query})
    return {"search_results": str(results)}

## Node 5: correct_syntax

GPT-4o-mini로 전사 결과를 분석하여 3가지를 제공:
1. Grammar corrections — 문법 오류 교정 + 설명
2. Vocabulary suggestions — 더 자연스러운 표현 제안
3. Sentence structure improvements — 문장 구조 개선

웹 검색 결과를 참고자료로 포함하여 교정 품질을 높인다.

In [ ]:
def correct_syntax(state: State):
    references = state.get("search_results", "")
    response = llm.invoke(
        [
            HumanMessage(
                content=(
                    "You are an English speaking test evaluator.\n"
                    "Analyze the following transcription from a picture description task.\n\n"
                    "Provide:\n"
                    "1. Grammar corrections with explanations\n"
                    "2. Vocabulary suggestions for more natural expression\n"
                    "3. Sentence structure improvements\n\n"
                    f"Transcription:\n{state['transcription']}\n\n"
                    f"Reference materials from web search:\n{references}\n\n"
                    "Use the reference materials to provide more accurate and "
                    "up-to-date suggestions. "
                    "Provide corrections in a clear, structured format."
                )
            )
        ]
    )
    return {"corrections": [response.content]}

## Node 6: recommend_ideal_answer

GPT-4o-mini (vision)에 생성된 이미지를 base64로 전달하여
TOEIC Speaking/OPIc 시험에 적합한 45-60초 분량의 모범답안을 생성한다.

In [ ]:
def recommend_ideal_answer(state: State):
    with open(state["image_dir"], "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")

    response = llm.invoke(
        [
            HumanMessage(
                content=[
                    {
                        "type": "text",
                        "text": (
                            "You are an English speaking test expert. "
                            "Generate an ideal 45-60 second picture description answer "
                            "for this image. Use natural, fluent English appropriate for "
                            "a TOEIC Speaking or OPIc test. Include descriptions of people, "
                            "actions, setting, and notable details."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_data}"
                        },
                    },
                ]
            )
        ]
    )
    return {"recommendations": [response.content]}

## Node 7: ask_regenerate + Conditional Edge

`interrupt()`로 사용자에게 재생성 여부를 묻는다.
- `True` → `correct_syntax`로 루프백 (교정 + 모범답안 재생성)
- `False` → `END` (종료)

재생성 시 `corrections`/`recommendations`는 `operator.add`로 누적되므로 이전 버전과 비교 가능.

In [ ]:
def ask_regenerate(state: State):
    answer = interrupt("Would you like to re-generate the correction and ideal answer?")
    return {"regenerate": answer}


def should_regenerate(state: State):
    if state.get("regenerate"):
        return "correct_syntax"
    return END

## Graph 구성

In [ ]:
memory = InMemorySaver()

graph_builder = StateGraph(State)

graph_builder.add_node("generate_image", generate_image)
graph_builder.add_node("record_voice", record_voice)
graph_builder.add_node("transcribe", transcribe)
graph_builder.add_node("search_references", search_references)
graph_builder.add_node("correct_syntax", correct_syntax)
graph_builder.add_node("recommend_ideal_answer", recommend_ideal_answer)
graph_builder.add_node("ask_regenerate", ask_regenerate)

graph_builder.add_edge(START, "generate_image")
graph_builder.add_edge("generate_image", "record_voice")
graph_builder.add_edge("record_voice", "transcribe")
graph_builder.add_edge("transcribe", "search_references")
graph_builder.add_edge("search_references", "correct_syntax")
graph_builder.add_edge("correct_syntax", "recommend_ideal_answer")
graph_builder.add_edge("recommend_ideal_answer", "ask_regenerate")
graph_builder.add_conditional_edges("ask_regenerate", should_regenerate)

graph = graph_builder.compile(checkpointer=memory, name="english-education-agent")

## Graph 시각화

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))